In [ ]:
!pip install -q google-genai sentence-transformers chromadb langchain-text-splitters pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [ ]:
import os
from google.colab import userdata ,files
from google import genai
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from openai import OpenAI

In [ ]:
api_key = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("O_R_A_K")

if not api_key:
    try:
        api_key = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        pass

os.environ["OPENROUTER_API_KEY"] = api_key

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
print(" Please upload one or more PDF files:")
uploaded = files.upload()

pdf_texts = []
for filename in uploaded.keys():
  if filename.endswith('.pdf'):
    reader = PdfReader(filename)
    text = ""
    for page_num, page in enumerate(reader.pages):
      page_text = page.extract_text()
      if page_text:
        text += f"\n --- Page {page_num +1} ---\n" + page_text
    pdf_texts.append(text)
    print(f" Loaded '{filename}' ({len(reader.pages)} pages).")

if not pdf_texts:
  raise ValueError("No valid PDF files uploaded. Please re-run and upload a .pdf")

full_pdf_content = "\n\n".join(pdf_texts)

 Please upload one or more PDF files:


Saving ANIKETCV.pdf to ANIKETCV (1).pdf
 Loaded 'ANIKETCV (1).pdf' (1 pages).


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = text_splitter.split_text(full_pdf_content)
print(f" Extracted and split document into {len(chunks)} text chunks.")


 Extracted and split document into 7 text chunks.


In [ ]:
print(" Loading embedding model and building vector index...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

#Reset collection for clean execution
try:
  chroma_client.delete_collection(name="pdf_rag_collection")
except Exception:
  pass

collection = chroma_client.create_collection(name="pdf_rag_collection")

#Embed chunks in batches
chunk_embeddings = embedder.encode(chunks).tolist()
chunks_ids = [f"doc_chunks_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=chunks_ids
)

print("PDF Vector Indexing Complete\n")

 Loading embedding model and building vector index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

PDF Vector Indexing Complete



In [ ]:
def ask_pdf(query: str):
    context_passages = retrieve_pdf_context(query, top_k=3)

    context_str = "\n".join(
        f"- {p}" for p in context_passages
    )

    prompt = f"""You are an intelligent document analysis assistant.
Answer the question using only the provided PDF context.

If the information is not contained within the provided context,
state clearly: "I cannot find this information in the provided PDF."

PDF Context:
{context_str}

Question: {query}

Answer:"""

    response = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning:free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    return answer, context_passages

In [ ]:
print("-" * 60)
print("PDF CHATBOT READY! Type your question below (or type 'exit' to quit).")
print("-" * 60)

while True:
    user_query = input("Ask a question about your PDF: ")
    if user_query.lower() in ["exit", "quit", "q"]:
        break

    if not user_query.strip():
        continue

    answer, context = ask_pdf(user_query)

    print("\n--- RETRIEVED PDF SNIPPETS ---")
    for i, snippet in enumerate(context, 1):
        print(f"[{i}] {snippet[:150]}...")

    print("\n--- NEMO RESPONSE ---")
    print(answer)
    print("-" * 60)

------------------------------------------------------------
PDF CHATBOT READY! Type your question below (or type 'exit' to quit).
------------------------------------------------------------
Ask a question about your PDF: Explain the whole resume and highlight the key points and weak points of it 

--- RETRIEVED PDF SNIPPETS ---
[1] --- Page 1 ---
     SUMMARY
SSC Sahakar Vidya Prasarak Mandal School ,Kalwa                                               
Driven and Innovative Full ...
[2] end and back-end development. Eager to learn, adapt, and contribute to cutting-edge teams!
ANIKET LAD
FULLSTACK DEVELOPER 
     TECHNICAL SKILLS
     ...
[3] recruiter job postings, and real-time application tracking. Integrated a chatbot for instant
user support and email job alerts. Ensured data security ...

--- NEMO RESPONSE ---
**Whole Resume Overview (based strictly on the provided PDF context):**

- **Name & Title:** Aniket Lad, Full Stack Developer  
- **Education (implied):** SSC from Sahakar